In [1]:
import torch
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")

# 1. Hardware Acceleration Check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Hardware Accelerator: {device.upper()}")

# 2. Generate Synthetic Explanation Data 
data = [
    ("I used a hash map to store the complements, allowing O(N) lookup time.", 2),
    ("By sorting the array first, I can use two pointers to find the target in O(N log N).", 2),
    ("I implemented dynamic programming with memoization to avoid recalculating overlapping subproblems.", 2),
    ("A breadth-first search is optimal here because we need the shortest path in an unweighted graph.", 2),
    ("I maintained a sliding window using two pointers to track the longest substring.", 2),
    ("I used a loop to check numbers, but I think the time complexity is O(N^2).", 1),
    ("I tried a hash map but I ended up storing all values instead of just the ones I need.", 1),
    ("I used recursion, but I forgot to add a base case so it might stack overflow on large inputs.", 1),
    ("It uses DFS to search the tree, though BFS might use less memory for this specific shape.", 1),
    ("I just guessed the answer and returned the first element.", 0),
    ("You can solve this by returning a random number from the array.", 0),
    ("I nested three for-loops to check every possible combination, it should be O(1) time.", 0),
    ("I imported the math library and called the built-in function to bypass the logic.", 0),
    ("I don't know how to solve this, I just wrote a print statement.", 0)
] * 20

df_nlp = pd.DataFrame(data, columns=["explanation", "label"])

# 3. Prepare the Hugging Face Dataset
hf_dataset = Dataset.from_pandas(df_nlp)
hf_dataset = hf_dataset.train_test_split(test_size=0.2, seed=42)

# 4. Tokenization
print("Downloading Tokenizer...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["explanation"], padding="max_length", truncation=True, max_length=64)

tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 5. Load Pre-trained Transformer 
print("Loading DistilBERT Model...")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', 
    num_labels=3
).to(device)

# 6. Define Evaluation Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    acc = accuracy_score(labels, preds)
    return {'Accuracy': acc, 'Precision': precision, 'Recall': recall, 'F1': f1}

# 7. Training Configuration
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    report_to="none",
    use_cpu=(device == "cpu")
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics
)

# 8. Train and Evaluate
print("Starting DistilBERT Training...")
trainer.train()

print("\nEvaluating DistilBERT Model...")
eval_results = trainer.evaluate()

print("\nModule 5: Transformer (DistilBERT) Metrics:")
print(f"Accuracy:  {eval_results['eval_Accuracy']:.4f}")
print(f"Precision: {eval_results['eval_Precision']:.4f}")
print(f"Recall:    {eval_results['eval_Recall']:.4f}")
print(f"F1 Score:  {eval_results['eval_F1']:.4f}")

c:\Users\mohit\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hardware Accelerator: CUDA


Map: 100%|██████████| 56/56 [00:00<00:00, 3183.79 examples/s]


Loading DistilBERT Model...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 701.67it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.098092,1.000000,1.000000,1.000000,1.000000
2,No log,0.014456,1.000000,1.000000,1.000000,1.000000
3,No log,0.010303,1.000000,1.000000,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]



Evaluating DistilBERT Model...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
No log,0.010303,3,1.000000,1.000000,1.000000,1.000000



Module 5: Transformer (DistilBERT) Metrics:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000


In [2]:
save_path = r"D:\skillmatrix\skill_matrix_ai\person_2\ml\models\explanation_classifier"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Saved DistilBERT model + tokenizer to {save_path}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.06it/s]

Saved DistilBERT model + tokenizer to D:\skillmatrix\skill_matrix_ai\person_2\ml\models\explanation_classifier
